# Documentation Chart Export

This notebook regenerates the PNG charts used in `docs/negative_price_analysis.md`.

Run it after the database has been populated and the PostgreSQL service is available. The notebook reads from the analysis views and writes PNG files to `docs/assets/charts/`.

PNG export uses Plotly's static image export, which requires `kaleido` to be installed in the active Python environment.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_database_config, load_env_file
from src.database import create_views, open_connection


load_env_file(PROJECT_ROOT / '.env')
config = load_database_config()

CHART_DIR = PROJECT_ROOT / 'docs' / 'assets' / 'charts'
CHART_DIR.mkdir(parents=True, exist_ok=True)

# Set to True if the SQL view definitions changed and should be refreshed before export.
REFRESH_VIEWS = False
if REFRESH_VIEWS:
    with open_connection(config) as connection:
        create_views(connection)

CHART_DIR

In [ ]:
px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = [
    '#2563eb',
    '#16a34a',
    '#dc2626',
    '#9333ea',
    '#f59e0b',
    '#0f766e',
]

CHART_WIDTH = 1100
CHART_HEIGHT = 650


def read_sql(query: str) -> pd.DataFrame:
    with open_connection(config) as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            rows = cursor.fetchall()
            columns = [column.name for column in cursor.description]

    return pd.DataFrame(rows, columns=columns)


def save_chart(fig, filename: str, width: int = CHART_WIDTH, height: int = CHART_HEIGHT) -> Path:
    output_path = CHART_DIR / filename
    fig.update_layout(
        width=width,
        height=height,
        margin=dict(l=75, r=35, t=85, b=70),
        title=dict(x=0.01, xanchor='left'),
        legend_title_text=None,
        font=dict(size=14),
    )
    fig.write_image(str(output_path), width=width, height=height, scale=2)
    return output_path


def add_bar_value_labels(fig, texttemplate: str = '%{text:,.0f}'):
    fig.update_traces(
        texttemplate=texttemplate,
        textposition='outside',
        cliponaxis=False,
        selector=dict(type='bar'),
    )
    fig.update_layout(uniformtext_minsize=9, uniformtext_mode='hide')
    fig.update_yaxes(rangemode='tozero')
    return fig


def format_percent_label(value: float, digits: int = 1) -> str:
    return f'{value:.{digits}f}%'


def format_share_label(value: float) -> str:
    return f'{value:.0%}'

## Negative price hours per year

In [ ]:
negative_hours_per_year = read_sql('''
    SELECT
        year,
        COUNT(*) FILTER (WHERE day_ahead_price < 0) AS negative_price_hours
    FROM hourly_system_features
    WHERE day_ahead_price IS NOT NULL
    GROUP BY year
    ORDER BY year;
''')

fig = px.bar(
    negative_hours_per_year,
    x='year',
    y='negative_price_hours',
    text='negative_price_hours',
    title='Negative Day-Ahead Price Hours per Year',
    labels={'year': 'Year', 'negative_price_hours': 'Negative price hours'},
)
fig.update_xaxes(dtick=1)
add_bar_value_labels(fig)
save_chart(fig, 'negative_price_hours_per_year.png')
fig.show()

## Negative price hours by month and hour

In [ ]:
negative_hours_month_hour = read_sql('''
    SELECT
        month,
        local_hour,
        COUNT(*) AS negative_price_hours
    FROM hourly_negative_price_features
    GROUP BY month, local_hour
    ORDER BY month, local_hour;
''')

full_month_hour_index = pd.MultiIndex.from_product(
    [range(1, 13), range(0, 24)], names=['month', 'local_hour']
)
heatmap_data = (
    negative_hours_month_hour.set_index(['month', 'local_hour'])
    .reindex(full_month_hour_index, fill_value=0)
    .reset_index()
)
heatmap_matrix = heatmap_data.pivot(
    index='month', columns='local_hour', values='negative_price_hours'
)

fig = px.imshow(
    heatmap_matrix,
    x=heatmap_matrix.columns,
    y=heatmap_matrix.index,
    aspect='auto',
    color_continuous_scale='Viridis',
    title='Negative Price Hours by Month and Hour of Day',
    labels={'x': 'Local hour', 'y': 'Month', 'color': 'Hours'},
)
fig.update_xaxes(dtick=1)
fig.update_yaxes(dtick=1)
save_chart(fig, 'negative_price_hours_month_hour.png')
fig.show()

## Negative price share by calendar type

In [ ]:
negative_share_by_calendar_type = read_sql('''
    WITH categorized AS (
        SELECT
            CASE
                WHEN is_holiday_de_lu = TRUE THEN 'holiday'
                WHEN is_holiday_de_lu = FALSE AND is_weekend = TRUE THEN 'weekend'
                ELSE 'weekday'
            END AS calendar_type,
            day_ahead_price
        FROM hourly_system_features
        WHERE day_ahead_price IS NOT NULL
    )
    SELECT
        calendar_type,
        COUNT(*) AS total_hours,
        COUNT(*) FILTER (WHERE day_ahead_price < 0) AS negative_price_hours,
        (
            100.0 * COUNT(*) FILTER (WHERE day_ahead_price < 0) / COUNT(*)
        )::double precision AS negative_price_share_pct
    FROM categorized
    GROUP BY calendar_type
    ORDER BY CASE calendar_type
        WHEN 'holiday' THEN 1
        WHEN 'weekend' THEN 2
        WHEN 'weekday' THEN 3
    END;
''')

negative_share_by_calendar_type['label'] = negative_share_by_calendar_type[
    'negative_price_share_pct'
].map(format_percent_label)

fig = px.bar(
    negative_share_by_calendar_type,
    x='calendar_type',
    y='negative_price_share_pct',
    text='label',
    title='Share of Negative Price Hours by Calendar Type',
    labels={
        'calendar_type': 'Calendar type',
        'negative_price_share_pct': 'Negative price hours (%)',
    },
    category_orders={'calendar_type': ['holiday', 'weekend', 'weekday']},
)
add_bar_value_labels(fig, '%{text}')
save_chart(fig, 'negative_price_share_calendar_type.png')
fig.show()

## Forecasted residual load by price group

In [ ]:
residual_load_by_price_group = read_sql(
    """
    SELECT
        CASE
            WHEN day_ahead_price < 0 THEN 'negative_price'
            ELSE 'non_negative_price'
        END AS price_group,
        forecasted_residual_load::double precision AS forecasted_residual_load
    FROM hourly_system_features
    WHERE day_ahead_price IS NOT NULL
      AND forecasted_residual_load IS NOT NULL;
    """
)

fig = px.box(
    residual_load_by_price_group,
    x="price_group",
    y="forecasted_residual_load",
    color="price_group",
    points="outliers",
    title="Forecasted Residual Load: Negative vs. Non-Negative Price Hours",
    labels={
        "price_group": "Price group",
        "forecasted_residual_load": "Forecasted residual load",
    },
    category_orders={
        "price_group": ["negative_price", "non_negative_price"]
    },
)

fig.update_layout(
    showlegend=True,
    legend_title_text="Price group",
)

y_min = residual_load_by_price_group["forecasted_residual_load"].min()
y_max = residual_load_by_price_group["forecasted_residual_load"].max()
y_padding = 0.05 * (y_max - y_min)

fig.update_yaxes(
    range=[
        min(y_min - y_padding, -22_000),
        y_max + y_padding,
    ],
    tick0=0,
    dtick=10_000,
    showgrid=True,
)

fig.write_image(
    PROJECT_ROOT / "docs" / "assets" / "charts" / "forecasted_residual_load_price_groups.png",
    width=1100,
    height=650,
    scale=2,
)

fig.show()

## Negative price share by forecasted residual-load decile

In [ ]:
negative_share_by_residual_load_decile = read_sql('''
    WITH ranked AS (
        SELECT
            day_ahead_price,
            NTILE(10) OVER (ORDER BY forecasted_residual_load) AS residual_load_decile
        FROM hourly_system_features
        WHERE day_ahead_price IS NOT NULL
          AND forecasted_residual_load IS NOT NULL
    )
    SELECT
        residual_load_decile,
        COUNT(*) AS total_hours,
        COUNT(*) FILTER (WHERE day_ahead_price < 0) AS negative_price_hours,
        (
            100.0 * COUNT(*) FILTER (WHERE day_ahead_price < 0) / COUNT(*)
        )::double precision AS negative_price_share_pct
    FROM ranked
    GROUP BY residual_load_decile
    ORDER BY residual_load_decile;
''')

negative_share_by_residual_load_decile['label'] = negative_share_by_residual_load_decile[
    'negative_price_share_pct'
].map(format_percent_label)

fig = px.bar(
    negative_share_by_residual_load_decile,
    x='residual_load_decile',
    y='negative_price_share_pct',
    text='label',
    title='Negative Price Share by Forecasted Residual Load Decile',
    labels={
        'residual_load_decile': 'Forecasted residual load decile',
        'negative_price_share_pct': 'Negative price hours (%)',
    },
)
fig.update_xaxes(dtick=1)
add_bar_value_labels(fig, '%{text}')
save_chart(fig, 'negative_price_share_residual_load_decile.png')
fig.show()

## Wind and solar share during negative price hours

In [ ]:
wind_solar_share_by_year = read_sql('''
    SELECT
        year,
        AVG(forecasted_solar_share_of_wind_solar)::double precision AS forecasted_solar_share,
        AVG(forecasted_wind_share_of_wind_solar)::double precision AS forecasted_wind_share
    FROM hourly_negative_price_features
    WHERE forecasted_solar_share_of_wind_solar IS NOT NULL
      AND forecasted_wind_share_of_wind_solar IS NOT NULL
    GROUP BY year
    ORDER BY year;
''')

wind_solar_share_long = wind_solar_share_by_year.melt(
    id_vars='year',
    value_vars=['forecasted_solar_share', 'forecasted_wind_share'],
    var_name='generation_type',
    value_name='share',
)
wind_solar_share_long['generation_type'] = wind_solar_share_long['generation_type'].replace(
    {
        'forecasted_solar_share': 'Solar share',
        'forecasted_wind_share': 'Wind share',
    }
)
wind_solar_share_long['label'] = wind_solar_share_long['share'].map(format_share_label)

fig = px.bar(
    wind_solar_share_long,
    x='year',
    y='share',
    color='generation_type',
    barmode='group',
    text='label',
    title='Average Wind and Solar Share during Negative Price Hours',
    labels={
        'year': 'Year',
        'share': 'Share of forecasted wind + solar',
        'generation_type': 'Generation type',
    },
)
fig.update_xaxes(dtick=1)
fig.update_yaxes(tickformat='.0%', range=[0, 1.08])
add_bar_value_labels(fig, '%{text}')
save_chart(fig, 'wind_solar_share_negative_hours.png')
fig.show()

## Negative price hours and events per year

In [ ]:
negative_hours_and_events_by_year = read_sql('''
    WITH hours AS (
        SELECT
            year,
            COUNT(*) AS negative_price_hours
        FROM hourly_negative_price_features
        GROUP BY year
    ),
    events AS (
        SELECT
            start_year AS year,
            COUNT(*) AS negative_price_events
        FROM negative_price_events
        GROUP BY start_year
    )
    SELECT
        COALESCE(hours.year, events.year) AS year,
        COALESCE(hours.negative_price_hours, 0) AS negative_price_hours,
        COALESCE(events.negative_price_events, 0) AS negative_price_events
    FROM hours
    FULL OUTER JOIN events
        ON hours.year = events.year
    ORDER BY year;
''')

negative_hours_and_events_long = negative_hours_and_events_by_year.melt(
    id_vars='year',
    value_vars=['negative_price_hours', 'negative_price_events'],
    var_name='metric',
    value_name='count',
)
negative_hours_and_events_long['metric'] = negative_hours_and_events_long['metric'].replace(
    {
        'negative_price_hours': 'Negative price hours',
        'negative_price_events': 'Negative price events',
    }
)

fig = px.bar(
    negative_hours_and_events_long,
    x='year',
    y='count',
    color='metric',
    barmode='group',
    text='count',
    title='Negative Price Hours and Events per Year',
    labels={'year': 'Year', 'count': 'Count', 'metric': 'Metric'},
)
fig.update_xaxes(dtick=1)
add_bar_value_labels(fig)
save_chart(fig, 'negative_hours_and_events_per_year.png')
fig.show()

## Event duration distribution

In [ ]:
event_duration_distribution = read_sql('''
    SELECT
        duration_hours,
        COUNT(*) AS event_count
    FROM negative_price_events
    WHERE duration_hours IS NOT NULL
    GROUP BY duration_hours
    ORDER BY duration_hours;
''')

fig = px.bar(
    event_duration_distribution,
    x='duration_hours',
    y='event_count',
    text='event_count',
    title='Distribution of Negative Price Event Duration',
    labels={
        'duration_hours': 'Event duration in hours',
        'event_count': 'Number of events',
    },
)
fig.update_xaxes(dtick=1)
add_bar_value_labels(fig)
save_chart(fig, 'event_duration_distribution.png')
fig.show()

## Yearly event duration

In [ ]:
yearly_event_duration_summary = read_sql('''
    SELECT
        start_year AS year,
        COUNT(*) AS negative_price_events,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY duration_hours)::double precision
            AS median_event_duration_hours
    FROM negative_price_events
    WHERE duration_hours IS NOT NULL
    GROUP BY start_year
    ORDER BY start_year;
''')

fig = go.Figure()
fig.add_bar(
    x=yearly_event_duration_summary['year'],
    y=yearly_event_duration_summary['negative_price_events'],
    text=yearly_event_duration_summary['negative_price_events'],
    textposition='outside',
    name='Negative price events',
    marker_color='#2563eb',
    cliponaxis=False,
)
fig.add_trace(
    go.Scatter(
        x=yearly_event_duration_summary['year'],
        y=yearly_event_duration_summary['median_event_duration_hours'],
        yaxis='y2',
        mode='lines+markers+text',
        text=yearly_event_duration_summary['median_event_duration_hours'].map(
            lambda value: f'{value:.1f} h'
        ),
        textposition='top center',
        name='Median duration',
        marker=dict(color='#dc2626', size=10),
        line=dict(color='#dc2626', width=3),
    )
)
fig.update_layout(
    title="Negative Price Events and Median Event Duration per Year",
    xaxis_title="Year",
    template="plotly_white",
    yaxis=dict(
        title="Negative price events",
        range=[0, 120],
        tick0=0,
        dtick=20,
        showgrid=True,
    ),
    yaxis2=dict(
        title="Median event duration in hours",
        overlaying="y",
        side="right",
        range=[0, 6],
        tick0=0,
        dtick=1,
        showgrid=False,
    ),
)
fig.update_xaxes(dtick=1)
save_chart(fig, 'negative_events_and_median_duration_by_year.png')
fig.show()


## Price-integral signal

In [ ]:
event_duration_vs_price_integral = read_sql('''
    SELECT
        start_year,
        duration_hours,
        ABS(price_integral_eur_per_mwh_hour)::double precision AS abs_price_integral
    FROM negative_price_events
    WHERE duration_hours IS NOT NULL
      AND price_integral_eur_per_mwh_hour IS NOT NULL;
''')
event_duration_vs_price_integral['start_year'] = event_duration_vs_price_integral[
    'start_year'
].astype(str)

fig = px.scatter(
    event_duration_vs_price_integral,
    x='duration_hours',
    y='abs_price_integral',
    color='start_year',
    title='Event Duration vs. Absolute Negative Price Integral',
    labels={
        'duration_hours': 'Event duration in hours',
        'abs_price_integral': 'Absolute price integral (sum of hourly EUR/MWh)',
        'start_year': 'Start year',
    },
)
fig.update_traces(marker=dict(size=9, opacity=0.75))
save_chart(fig, 'event_duration_vs_price_integral.png')
fig.show()

## Usable events by minimum runtime and year

In [ ]:
usable_events_by_runtime_year = read_sql('''
    WITH minimum_runtimes(minimum_runtime_hours) AS (
        VALUES (1), (2), (4), (6)
    ),
    years AS (
        SELECT DISTINCT start_year AS year
        FROM negative_price_events
    ),
    total_events AS (
        SELECT
            start_year AS year,
            COUNT(*) AS total_negative_price_events
        FROM negative_price_events
        GROUP BY start_year
    )
    SELECT
        years.year,
        minimum_runtimes.minimum_runtime_hours,
        COUNT(negative_price_events.event_id) AS usable_negative_price_events,
        (
            100.0 * COUNT(negative_price_events.event_id)
            / NULLIF(total_events.total_negative_price_events, 0)
        )::double precision AS usable_event_share_pct
    FROM years
    CROSS JOIN minimum_runtimes
    LEFT JOIN total_events
        ON total_events.year = years.year
    LEFT JOIN negative_price_events
        ON negative_price_events.start_year = years.year
       AND negative_price_events.duration_hours >= minimum_runtimes.minimum_runtime_hours
    GROUP BY
        years.year,
        minimum_runtimes.minimum_runtime_hours,
        total_events.total_negative_price_events
    ORDER BY
        years.year,
        minimum_runtimes.minimum_runtime_hours;
''')

usable_events_by_runtime_year['minimum_runtime_label'] = usable_events_by_runtime_year[
    'minimum_runtime_hours'
].astype(str) + ' h minimum runtime'

fig = px.bar(
    usable_events_by_runtime_year,
    x='year',
    y='usable_negative_price_events',
    color='minimum_runtime_label',
    barmode='group',
    text='usable_negative_price_events',
    title='Usable Negative Price Events by Minimum Runtime and Year',
    labels={
        'year': 'Year',
        'usable_negative_price_events': 'Usable negative price events',
        'minimum_runtime_label': 'Minimum runtime',
    },
)
fig.update_xaxes(dtick=1)
add_bar_value_labels(fig)
save_chart(fig, 'usable_events_by_minimum_runtime_by_year.png', width=1250, height=700)
fig.show()

## Exported files

In [ ]:
sorted(path.name for path in CHART_DIR.glob('*.png'))